# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
spaceship.shape

(8693, 14)

**Check for data types**

In [4]:
spaceship.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

**Check for missing values**

In [5]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
spaceship_clean = spaceship.dropna().copy()

print("Before:", len(spaceship))
print("After:", len(spaceship_clean))
print("Removed:", len(spaceship) - len(spaceship_clean))

Before: 8693
After: 6606
Removed: 2087


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]

spaceship_clean["Cabin"].value_counts()

Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

- Drop PassengerId and Name

In [8]:
spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])

X = spaceship_clean.drop(columns="Transported")
y = spaceship_clean["Transported"]

- For non-numerical columns, do dummies.

In [9]:
#your code here

**Perform Train Test Split**

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

print("Practice passengers:", len(X_train))
print("Test passengers:", len(X_test))

Practice passengers: 5284
Test passengers: 1322


In [10]:
from sklearn.preprocessing import OneHotEncoder

# Find the columns that need encoding, including True/False columns.
category_columns = X_train.select_dtypes(exclude="number").columns.tolist()

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Learn the categories from the training passengers only.
encoder.fit(X_train[category_columns])

# Use the same encoder to turn both sets into 0/1 columns.
column_names = encoder.get_feature_names_out(category_columns)

train_encoded = pd.DataFrame(
    encoder.transform(X_train[category_columns]),
    columns=column_names,
    index=X_train.index
)

test_encoded = pd.DataFrame(
    encoder.transform(X_test[category_columns]),
    columns=column_names,
    index=X_test.index
)

# Put the original number columns and the new 0/1 columns together.
X_train_ready = pd.concat(
    [X_train.drop(columns=category_columns), train_encoded],
    axis=1
)

X_test_ready = pd.concat(
    [X_test.drop(columns=category_columns), test_encoded],
    axis=1
)

X_train_ready.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
2700,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1227,37.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1679,28.0,0.0,11026.0,0.0,0.0,12.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4910,46.0,1.0,0.0,898.0,0.0,96.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3934,40.0,0.0,0.0,1.0,293.0,667.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [11]:
from sklearn.preprocessing import OneHotEncoder

# 1. Find the columns that need to become 0/1 columns.
category_columns = X_train.select_dtypes(exclude="number").columns.tolist()

# 2. Learn the categories from the training data only.
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train[category_columns])

# 3. Use the same encoder on both sets.
encoded_columns = encoder.get_feature_names_out(category_columns)

train_encoded = pd.DataFrame(
    encoder.transform(X_train[category_columns]),
    columns=encoded_columns,
    index=X_train.index
)

test_encoded = pd.DataFrame(
    encoder.transform(X_test[category_columns]),
    columns=encoded_columns,
    index=X_test.index
)

# 4. Put the original number columns beside the new 0/1 columns.
X_train_ready = pd.concat(
    [X_train.drop(columns=category_columns), train_encoded], axis=1
)

X_test_ready = pd.concat(
    [X_test.drop(columns=category_columns), test_encoded], axis=1
)

X_train_ready.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
2700,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1227,37.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1679,28.0,0.0,11026.0,0.0,0.0,12.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4910,46.0,1.0,0.0,898.0,0.0,96.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3934,40.0,0.0,0.0,1.0,293.0,667.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [12]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

- Evaluate your model's performance. Comment it

In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.1%}")

pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[False, True]),
    index=["Actually not transported", "Actually transported"],
    columns=["Predicted not transported", "Predicted transported"]
)

Test accuracy: 79.0%


,Predicted not transported,Predicted transported
Actually not transported,527,129
Actually transported,149,517


In [ ]:
#After one-hot encoding the categorical features, the KNN model achieved 79.0% test accuracy. It correctly identified 527 passengers who were not transported and 517 who were transported. It also incorrectly predicted that 129 passengers were transported and missed 149 who actually were.